# R-Training Pipeline Runner

This notebook runs the final R-Training workflow:

- **Adaptive-filter branch** for `texas`, `cornell`, `actor`, `chameleon`, `squirrel`.
  - Uses `tuned_params/<dataset>.json`.
  - Runs once per dataset/label-rate because the filter params are fixed by adaptive-filter params.

- **Base 8-filter branch** for all other configured datasets.
  - Uses fixed `sample_seed = 42`.
  - Runs once per dataset/label-rate/filter.
  - You choose which base filters to run in the configuration cell.

Results are saved after every task, so the notebook can resume safely.

## 1. Install Dependencies

Run this cell first on Colab or any fresh environment. It installs PyTorch Geometric compiled dependencies such as `torch_sparse`.


In [ ]:
# Install dependencies for Colab / fresh environments.
# Run this before importing the project.

import sys
import subprocess
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)

requirements = [
    'numpy',
    'scipy',
    'pandas',
    'scikit-learn',
    'tqdm',
    'matplotlib',
    'networkx',
    'torch-geometric',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])

torch_version = torch.__version__.split('+')[0]
cuda_tag = torch.__version__.split('+')[1] if '+' in torch.__version__ else 'cpu'
pyg_wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel URL:', pyg_wheel_url)

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv',
    '-f', pyg_wheel_url,
])

# Quick import check.
import torch_geometric
import torch_sparse
print('torch_geometric:', torch_geometric.__version__)
print('torch_sparse import: OK')


## 2. Project Setup

Set `PROJECT_ROOT` to the project folder. On Colab, mount Drive first and set the path to your uploaded project.

In [ ]:
from pathlib import Path
import os

# Optional on Colab:
# from google.colab import drive
# drive.mount('/content/drive')

# If this notebook is opened from notebooks/, move one level up to the project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Colab fallback for your renamed project folder.
if not (PROJECT_ROOT / 'pipeline.py').exists():
    candidate = Path('/content/drive/MyDrive/R-Training')
    if (candidate / 'pipeline.py').exists():
        PROJECT_ROOT = candidate

os.chdir(PROJECT_ROOT)
print('Project root:', Path.cwd())
print('pipeline.py exists:', Path('pipeline.py').exists())


## 3. Configuration

Edit this cell to choose datasets, label rates, base filters, and run count.

In [ ]:
MULTICLASS_HETEROPHILIC_DATASETS = ["texas", "cornell", "actor", "chameleon", "squirrel"]

HOMOPHILIC_DATASETS = [
    "cora", "citeseer", "pubmed",
    "photo", "computers",
]

BINARY_HETEROPHILIC_DATASETS = [
    "minesweeper", "tolokers", "questions",
]

# Choose what to run.
DATASETS = HOMOPHILIC_DATASETS + BINARY_HETEROPHILIC_DATASETS + MULTICLASS_HETEROPHILIC_DATASETS

# Label-rate splits to run.
LABEL_RATES = [0.005, 0.01, 0.02, 0.03, 0.04, 0.05]

# For base-filter datasets only. Adaptive-filter datasets ignore this and use tuned_params/*.json.
BASE_FILTERS_TO_RUN = [
    "g_0",
    # "g_1",
    # "g_2",
    # "g_3",
    # "g_low_pass",
    # "g_high_pass",
    # "g_band_pass",
    # "g_band_rejection",
]
BASE_FILTER_DEGREE = 8

NUM_RUNS = 10
OVERWRITE = False
VERBOSE = False
EVALUATE_NEW_LABELS = False

OUTPUT_ROOT = Path("outputs/r_training")
RAW_DIR = OUTPUT_ROOT / "raw"
SUMMARY_CSV = OUTPUT_ROOT / "summary.csv"
SUMMARY_JSON = OUTPUT_ROOT / "summary.json"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Datasets:', DATASETS)
print('Label rates:', LABEL_RATES)
print('Base filters:', BASE_FILTERS_TO_RUN)
print('Num runs:', NUM_RUNS)
print('Overwrite:', OVERWRITE)

## 4. Import Pipeline

This imports the project API. The notebook does not duplicate the algorithm logic.

In [ ]:
import json
import pandas as pd
from tqdm.auto import tqdm

from pipeline import run_one_config, get_r_training_branch

print('Pipeline imported successfully.')

## 5. Build Task List

Adaptive-filter datasets produce one task per label rate. Standard datasets produce one task per label rate per selected filter.

In [ ]:
def rate_key(rate):
    return f"{float(rate):.4f}"


def task_output_path(dataset, label_rate, filter_name):
    safe_filter = str(filter_name).replace('/', '_')
    return RAW_DIR / dataset / f"rate_{rate_key(label_rate)}__filter_{safe_filter}.json"


def build_tasks():
    tasks = []
    for dataset in DATASETS:
        r_training_branch = get_r_training_branch(dataset)
        for label_rate in LABEL_RATES:
            if r_training_branch == "adaptive_filters":
                tasks.append({
                    "dataset": dataset,
                    "r_training_branch": r_training_branch,
                    "label_rate": float(label_rate),
                    "base_filter": "adaptive_filters",
                    "r_training_filter_cfg": None,
                })
            else:
                for filter_name in BASE_FILTERS_TO_RUN:
                    tasks.append({
                        "dataset": dataset,
                        "r_training_branch": r_training_branch,
                        "label_rate": float(label_rate),
                        "base_filter": filter_name,
                        "r_training_filter_cfg": {
                            "filter_name": filter_name,
                            "degree": BASE_FILTER_DEGREE,
                        },
                    })
    return tasks


tasks = build_tasks()
print('Total tasks:', len(tasks))
pd.DataFrame(tasks).head(20)

## 6. Run Experiments

Each task is saved immediately as JSON. Existing task outputs are skipped unless `OVERWRITE=True`.

In [ ]:
def to_summary_row(result, base_filter):
    return {
        "dataset": result["dataset"],
        "r_training_branch": result["r_training_branch"],
        "label_rate": result["label_rate"],
        "sample_seed": result["sample_seed"],
        "base_filter": base_filter,
        "initial_labels": result["initial_labels"],
        "r_training_added_labels": result["r_training_added_labels"],
        "r_training_label_acc": result["r_training_label_acc"],
        "acc_mean": result["acc_mean"],
        "acc_std": result["acc_std"],
        "macro_f1_mean": result["macro_f1_mean"],
        "macro_f1_std": result["macro_f1_std"],
        "roc_auc_mean": result.get("roc_auc_mean"),
        "roc_auc_std": result.get("roc_auc_std"),
        "model_cfg": json.dumps(result["model_cfg"], sort_keys=True),
        "r_training_filter_cfg": json.dumps(result["r_training_filter_cfg"], sort_keys=True),
    }


completed = []
skipped = []
failed = []

for task in tqdm(tasks, desc="pipeline tasks"):
    out_path = task_output_path(task["dataset"], task["label_rate"], task["base_filter"])
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and not OVERWRITE:
        skipped.append(task)
        continue

    try:
        result = run_one_config(
            dataset_name=task["dataset"],
            label_rate=task["label_rate"],
            r_training_filter_cfg=task["r_training_filter_cfg"],
            num_runs_override=NUM_RUNS,
            verbose=VERBOSE,
            evaluate_new_labels=EVALUATE_NEW_LABELS,
            use_r_training=True,
        )
        result["base_filter"] = task["base_filter"]
        result["num_runs"] = NUM_RUNS

        with out_path.open("w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        completed.append(task)

    except Exception as exc:
        failed_task = dict(task)
        failed_task["error"] = repr(exc)
        failed.append(failed_task)
        print("FAILED:", failed_task)
        raise

print('Completed:', len(completed))
print('Skipped:', len(skipped))
print('Failed:', len(failed))

## 7. Build Summary Tables

This scans all saved raw JSON files and creates CSV/JSON summaries.

In [ ]:
rows = []
for path in sorted(RAW_DIR.glob("*/*.json")):
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
    rows.append(to_summary_row(result, result.get("base_filter", "adaptive_filters")))

summary = pd.DataFrame(rows)
if not summary.empty:
    summary = summary.sort_values(["dataset", "label_rate", "base_filter"]).reset_index(drop=True)
    summary.to_csv(SUMMARY_CSV, index=False)
    summary.to_json(SUMMARY_JSON, orient="records", indent=2)

print('Rows:', len(summary))
print('Saved CSV:', SUMMARY_CSV)
print('Saved JSON:', SUMMARY_JSON)
summary

## 8. Accuracy Pivot

Quick view of accuracy by dataset, rate, and filter.

In [ ]:
if summary.empty:
    print('No summary rows yet.')
else:
    display_cols = ["dataset", "r_training_branch", "label_rate", "base_filter", "acc_mean", "acc_std", "macro_f1_mean", "roc_auc_mean", "r_training_added_labels", "r_training_label_acc"]
    display(summary[display_cols])

    pivot = summary.pivot_table(
        index=["dataset", "r_training_branch", "base_filter"],
        columns="label_rate",
        values="acc_mean",
        aggfunc="first",
    )
    display(pivot)